# FlightOnTime — Production ML Pipeline (v6.0)

**Author:** Data Science Team  
**Stack:** CatBoost + Hydra + MLflow + Stratified K-Fold CV  
**Version:** 6.0.0 (Production-Ready Architecture)

---

## From Notebook to Production Pipeline

This notebook documents the **production architecture** of FlightOnTime. Unlike v5.0 (monolithic notebook),
here we separate concerns into reusable modules, externalize all configuration via **Hydra/OmegaConf**,
and track every experiment in **MLflow**.

### What Changed?

| Aspect | v5.0 (Notebook) | v6.0 (Production) |
|:--------|:----------------|:--------------------|
| Configuration | Hardcoded in code | YAML via Hydra |
| Experiments | print() to stdout | MLflow tracking |
| Validation | None | Pandera schemas |
| CV | None (single holdout) | Stratified K-Fold |
| Code | All in one notebook | Modules in `src/` |
| Model | Manual .joblib | MLflow Registry + .joblib |
| API | Coupled to artifact format | Health check + dual loading |

# 1. Setup & Configuration

We load all settings via Hydra/OmegaConf — **zero hardcoding**.

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

plt.style.use('ggplot')
pd.set_option('display.max_columns', None)
%matplotlib inline

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# HYDRA CONFIG — Single Source of Truth
# ═══════════════════════════════════════════════════════════════
# In a notebook we compose the config manually (Hydra CLI works in scripts).
# This is equivalent to: python -m src.models.train

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

# Clear any existing Hydra state
GlobalHydra.instance().clear()

config_dir = os.path.join(PROJECT_ROOT, 'config')
with initialize_config_dir(version_base=None, config_dir=config_dir):
    cfg = compose(config_name='config')

print('=== Hydra Configuration Loaded ===')
print(OmegaConf.to_yaml(cfg))

### 1.1 The Hydra Advantage

Notice: **no magic numbers in the code**. Everything comes from `config.yaml` and its sub-files.
To change the model from 500 to 1000 iterations, just edit `config/model/catboost.yaml`
or run from the CLI:

```bash
python -m src.models.train model.iterations=1000
```

Without touching **a single line of code**.

# 2. Data Loading & Cleaning (Modular)

We call `src.data.preprocess` — all cleaning logic is encapsulated and testable.
Every rule (which columns to drop, outlier thresholds, target definition) is driven by the config.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s | %(message)s')

from src.data.preprocess import load_raw, clean

# Load
df_raw = load_raw(cfg)
print(f'Raw: {df_raw.shape}')

# Clean (all rules driven by cfg.data)
df = clean(df_raw, cfg)
print(f'Clean: {df.shape}')

In [ ]:
# Quick sanity check
print('Target distribution:')
print(df['target'].value_counts())
print(f'\nDelay rate: {df["target"].mean():.1%}')

# 3. Feature Engineering (Modular)

Haversine distance, temporal features, holidays, weather — all in `src.features.engineer`.

The feature list and categorical column definitions come from `config/features/weather_aware.yaml`.

In [ ]:
from src.features.engineer import build_features, select_features, build_airport_coords

# Extract airport coordinates (before renaming)
airport_coords = build_airport_coords(df_raw)
print(f'Airport coordinate lookup: {len(airport_coords)} airports')

# Full feature engineering
df = build_features(df, cfg)

# Select model features
X, y, feature_names, cat_feature_names = select_features(df, cfg)
print(f'\nFeature matrix: {X.shape}')
print(f'Features: {feature_names}')
print(f'Categorical: {cat_feature_names}')
X.head()

# 4. Data Validation (Schema Check)

Before training, we validate the feature matrix against a Pandera schema.
This catches issues like new airlines with null values, negative precipitation, etc.

In [ ]:
from src.data.validate import validate

df_to_validate = pd.concat([X, y], axis=1)
validated = validate(df_to_validate, cfg)
print('Validation passed!')

# 5. Train/Test Split + Stratified K-Fold CV

### Why There Is No Data Leakage in Random Splitting

Each flight's delay is **conditionally independent** given the features available at prediction time
(route, scheduled departure, weather forecast). We do not use any feature that depends on future flights —
therefore, knowing the delay of a flight in March does not "leak" information about a flight in January.

A temporal split concern would be valid if we had features like "average delay in the last 24 hours"
(concept drift / autocorrelation), but that is not the case here. All our features are available
before the flight departs.

### Stratified K-Fold CV

Given the severe class imbalance (88.4% on-time vs 11.6% delayed), we use **Stratified K-Fold**
to ensure each fold preserves the class ratio. This gives a more robust and reliable performance
estimate than a single holdout split.

In [ ]:
from sklearn.model_selection import train_test_split

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=cfg.split.test_size,
    random_state=cfg.split.random_state,
    stratify=y if cfg.split.stratify else None,
)

print(f'Train: {len(X_train):,} ({y_train.mean():.1%} delayed)')
print(f'Test:  {len(X_test):,} ({y_test.mean():.1%} delayed)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STRATIFIED K-FOLD CROSS-VALIDATION
# ═══════════════════════════════════════════════════════════════
from src.models.train import run_cv

cv_results = run_cv(X_train, y_train, cfg, cat_feature_names)

print('\n=== Cross-Validation Results ===')
for k, v in cv_results.items():
    print(f'  {k}: {v}')

# 6. Model Training & MLflow Tracking

We train the model on the full training set, evaluate on the holdout, and log everything to MLflow.
Every parameter, metric, and artifact is recorded — so we can compare runs, reproduce results,
and trace any model back to its exact configuration.

In [ ]:
import mlflow
import mlflow.catboost
from catboost import CatBoostClassifier
from sklearn.metrics import (
    classification_report, recall_score, precision_score,
    f1_score, fbeta_score, accuracy_score, roc_auc_score,
)
from omegaconf import OmegaConf

# ── MLflow Setup ──
mlflow.set_tracking_uri(cfg.mlflow.tracking_uri)
mlflow.set_experiment(cfg.mlflow.experiment_name)

with mlflow.start_run(run_name=f'{cfg.model.name}_notebook') as run:
    # Log config
    model_params = OmegaConf.to_container(cfg.model, resolve=True)
    model_params.pop('name', None)
    mlflow.log_params({f'model.{k}': v for k, v in model_params.items()})
    mlflow.log_params({
        'cv.n_splits': cfg.cv.n_splits,
        'split.test_size': cfg.split.test_size,
        'threshold': cfg.threshold.value,
        'n_samples': len(X),
        'n_features': len(feature_names),
    })
    
    # Log CV results
    mlflow.log_metrics(cv_results)
    
    # ── Train Final Model ──
    print('Training CatBoost on training set ...')
    model = CatBoostClassifier(
        **model_params,
        cat_features=cat_feature_names,
    )
    model.fit(X_train, y_train)
    
    # ── Holdout Evaluation ──
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    holdout_metrics = {
        'holdout_recall': recall_score(y_test, y_pred),
        'holdout_precision': precision_score(y_test, y_pred),
        'holdout_f1': f1_score(y_test, y_pred),
        'holdout_f2': fbeta_score(y_test, y_pred, beta=2),
        'holdout_accuracy': accuracy_score(y_test, y_pred),
        'holdout_roc_auc': roc_auc_score(y_test, y_proba),
    }
    mlflow.log_metrics(holdout_metrics)
    
    print('\n=== Holdout Test Results ===')
    for k, v in holdout_metrics.items():
        print(f'  {k}: {v:.4f}')
    
    print('\n' + classification_report(y_test, y_pred, target_names=['On-Time', 'Delayed']))
    
    RUN_ID = run.info.run_id
    print(f'\nMLflow Run ID: {RUN_ID}')

# 7. Threshold Optimization

We search for the F2-Score optimal threshold (which gives double weight to Recall),
then apply the **business safety override** at 0.35.

The rationale: a false alarm ("flight might be delayed" when it's not) costs far less
than a missed delay (passenger misses their connection). So we trade precision for recall.

In [ ]:
from src.models.evaluate import optimize_threshold, log_confusion_matrix, log_feature_importance

with mlflow.start_run(run_id=RUN_ID):
    # Threshold analysis
    threshold_results = optimize_threshold(y_test, y_proba, cfg)
    
    print(f'\nMath-optimal threshold: {threshold_results["math_threshold"]:.3f}')
    print(f'Business threshold:     {threshold_results["business_threshold"]:.3f}')
    print(f'Recall at business:     {threshold_results["recall"]:.1%}')
    print(f'Precision at business:  {threshold_results["precision"]:.1%}')
    
    mlflow.log_metrics({
        'optimal_threshold': threshold_results['business_threshold'],
        'threshold_recall': threshold_results['recall'],
        'threshold_precision': threshold_results['precision'],
    })

In [ ]:
# ── Confusion Matrix & Feature Importance (logged to MLflow) ──
with mlflow.start_run(run_id=RUN_ID):
    log_confusion_matrix(y_test, y_proba, threshold_results['business_threshold'])
    log_feature_importance(model, feature_names)
    print('Plots logged to MLflow artifacts.')

# 8. Production Export

Re-train on **all** data (train + test) to maximize the information available to the production model,
then export via two channels:

1. **MLflow Model Registry** — for programmatic loading and lifecycle management (Staging/Production/Archived)
2. **Joblib artifact** — for Docker and offline deployment environments

In [ ]:
from src.models.export import export_production_artifact

# Re-train on full data
print('Re-training on full dataset for production …')
model_params_clean = OmegaConf.to_container(cfg.model, resolve=True)
model_params_clean.pop('name', None)

model_full = CatBoostClassifier(
    **model_params_clean,
    cat_features=cat_feature_names,
)
model_full.fit(X, y, verbose=0)

with mlflow.start_run(run_id=RUN_ID):
    # Register in MLflow
    mlflow.catboost.log_model(
        model_full,
        artifact_path='model',
        registered_model_name='FlightOnTime' if cfg.mlflow.register_model else None,
    )
    
    # Export joblib
    path = export_production_artifact(
        model=model_full,
        feature_names=feature_names,
        cat_feature_names=cat_feature_names,
        airport_coords=airport_coords,
        threshold=threshold_results['business_threshold'],
        metrics=holdout_metrics,
        cv_metrics=cv_results,
        cfg=cfg,
    )
    
    print(f'\nProduction artifact: {path}')
    print(f'MLflow Run: {RUN_ID}')
    print('Model registered in MLflow Model Registry.')

# 9. API Simulation (Digital Twin)

We simulate the exact behavior of the `/predict` endpoint using the exported model.
This validates the full inference pipeline end-to-end before deployment.

In [ ]:
import holidays
from IPython.display import display, Markdown

THRESHOLD_BIZ = threshold_results['business_threshold']

def simulate_api(request: dict) -> float:
    """Simulate the /predict endpoint locally."""
    dt = pd.to_datetime(request['departure_datetime'])
    is_hol = 1 if dt.date() in holidays.Brazil() else 0
    
    input_df = pd.DataFrame([{
        'companhia': str(request['airline']),
        'origem': str(request['origin']),
        'destino': str(request['destination']),
        'distancia_km': float(request['distance_km']),
        'hora': dt.hour,
        'dia_semana': dt.dayofweek,
        'mes': dt.month,
        'is_holiday': is_hol,
        'precipitation': float(request.get('precipitation', 0.0)),
        'wind_speed': float(request.get('wind_speed', 5.0)),
        'clima_imputado': 0,
    }])
    
    prob = model_full.predict_proba(input_df[feature_names])[0][1]
    
    if prob < THRESHOLD_BIZ:
        emoji, status = '🟢', 'ON TIME'
    elif prob < 0.70:
        emoji, status = '🟡', 'PREVENTIVE ALERT'
    else:
        emoji, status = '🔴', 'LIKELY DELAYED'
    
    display(Markdown(
        f'### {emoji} {status} — Probability: **{prob:.1%}**\n'
        f'* Flight: {request["airline"]} | {request["origin"]} -> {request["destination"]}\n'
        f'* Datetime: {dt.strftime("%Y-%m-%d %H:%M")} ({"Holiday" if is_hol else "Regular day"})\n'
        f'* Weather: Rain {request.get("precipitation", 0)}mm | Wind {request.get("wind_speed", 5)}km/h\n---'
    ))
    return prob

# Scenario 1: Clear weather
p1 = simulate_api({
    'airline': 'GOL', 'origin': 'Congonhas', 'destination': 'Santos Dumont',
    'departure_datetime': '2025-11-10T08:00:00', 'distance_km': 366,
    'precipitation': 0.0, 'wind_speed': 5.0,
})

# Scenario 2: Storm conditions
p2 = simulate_api({
    'airline': 'GOL', 'origin': 'Congonhas', 'destination': 'Santos Dumont',
    'departure_datetime': '2025-11-10T08:00:00', 'distance_km': 366,
    'precipitation': 22.0, 'wind_speed': 48.0,
})

print(f'Weather impact: +{(p2-p1)*100:.1f} percentage points')

# 10. How to Use in Production

## CLI Training (Hydra)
```bash
# Default config
python -m src.models.train

# Override model hyperparameters
python -m src.models.train model=catboost_tuned

# Override individual parameters
python -m src.models.train model.iterations=1000 cv.n_splits=10

# Change threshold strategy
python -m src.models.train threshold.value=0.40
```

## Serving API
```bash
# Load from joblib artifact
MODEL_SOURCE=joblib uvicorn src.serving.app:app --reload

# Load from MLflow Model Registry
MODEL_SOURCE=mlflow uvicorn src.serving.app:app --reload
```

## MLflow Dashboard
```bash
mlflow ui --backend-store-uri mlruns
# Open http://localhost:5000 to compare experiments
```